# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202409_Hurricane_Helene'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'sentinel1'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 46 .tif files in the S3 bucket.


['drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240914_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240919_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240921_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240926_N_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240926_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240914T233731_DVR_RTC20_G_gpufed_2C0F_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240914T233756_DVR_RTC20_G_gpufed_9ED7_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232244_DVR_RTC20_G_gpufed_C767_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232

## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 127
  - Total size: 4.85 GB

📁 Cached files (first 10):
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif (44.0 MB)
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif (11.0 MB)
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024263.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif (44.0 MB)
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024263.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif (11.0 MB)
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024264.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif (44.0 MB)
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024264.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif (11.0 MB)
  - drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024265.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif (44.0 MB)
  - drcs_activations

(127, 5206263769)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys


['drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240914_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240919_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240921_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240926_N_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240926_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240914T233731_DVR_RTC20_G_gpufed_2C0F_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240914T233756_DVR_RTC20_G_gpufed_9ED7_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232244_DVR_RTC20_G_gpufed_C767_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232

In [11]:
def convert_sentinel_datetime(datetime_str):
    """
    Convert Sentinel datetime format to ISO 8601 format with UTC timezone.
    
    Args:
        datetime_str: String like '20240430T002653'
    
    Returns:
        String like '2024-04-30T00:26:53Z'
    """
    # Extract components
    year = datetime_str[0:4]
    month = datetime_str[4:6]
    day = datetime_str[6:8]
    hour = datetime_str[9:11]
    minute = datetime_str[11:13]
    second = datetime_str[13:15]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}T{hour}:{minute}:{second}Z"

# Test
datetime_str = '20240430T002653'
result = convert_sentinel_datetime(datetime_str)
print(result)  # 2024-04-30T00:26:53Z

2024-04-30T00:26:53Z


In [31]:
def create_cog_filename_WM(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    from pathlib import Path
    
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Check if it's a simple format (S1A_YYYYMMDD_rgb)
    if len(fsplit) == 3 and len(fsplit[1]) == 8 and fsplit[1].isdigit():
        # Simple format: S1A_20240914_rgb
        satellite = fsplit[0]
        date_str = fsplit[1]
        file_type = fsplit[2]
        
        # Format date from YYYYMMDD to YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        cog_filename = f'{EVENT_NAME}_{satellite}_{file_type}_{formatted_date}_day.tif'
    
    # Check if it's a complex format with timestamp
    elif len(fsplit) >= 8 and 'T' in fsplit[2]:
        # Complex format: S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_WM
        satellite = fsplit[0]
        mode = fsplit[1]
        datetime_str = fsplit[2]  # 20240916T232218
        
        # Use the convert_sentinel_datetime function for full timestamp
        formatted_datetime = convert_sentinel_datetime(datetime_str)
        
        # Check if "reclassified" is in the filename
        if "reclassified" in f2:
            # Find where reclassified is and include it
            processing_info = '_'.join(fsplit[3:8])  # DVR_RTC20_G_gpufed_9521
            file_type = '_'.join(fsplit[8:])  # reclassified_WM
        else:
            # Regular format without reclassified
            processing_info = '_'.join(fsplit[3:8])  # DVR_RTC20_G_gpufed_9521
            file_type = fsplit[-1]  # WM or rgb
        
        cog_filename = f'{EVENT_NAME}_{satellite}_{mode}_{processing_info}_{file_type}_{formatted_datetime}.tif'
    
    else:
        # Fallback for unexpected formats
        cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    
    return cog_filename

filter_str = '_WM'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
print(len(filter_))

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_WM(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
24
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_2C0F_WM_2024-09-14T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9ED7_WM_2024-09-14T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9521_WM_2024-09-16T23:22:18Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_C767_WM_2024-09-16T23:22:44Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_FCE1_WM_2024-09-16T23:23:09Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_4389_WM_2024-09-16T23:23:34Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_8A7C_WM_2024-09-19T23:45:49Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9A50_WM_2024-09-19T23:46:17Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_7FC6_reclassified_WM_2024-09-26T23:37:06Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_E134_reclassified_WM_2024-09-26T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_E28C_reclassified_WM_2024-09-26T23:37:56Z.tif
  

In [13]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_WM, 
                                target_dir = "Sentinel-1/WM", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_2C0F_WM_2024-09-14T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9ED7_WM_2024-09-14T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9521_WM_2024-09-16T23:22:18Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_C767_WM_2024-09-16T23:22:44Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_FCE1_WM_2024-09-16T23:23:09Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_4389_WM_2024-09-16T23:23:34Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_8A7C_WM_2024-09-19T23:45:49Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9A50_WM_2024-09-19T23:46:17Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_7FC6_WM_2024-09-26T23:37:06Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_E134_WM_2024-09-26T23:37:31Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_E28C_WM_2024-09-26T23:37:56Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_g

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpchyw7kyd_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp_cz9c3_g.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=4, center sample non-zero=1666/1000000
            Estimated data coverage: 0.1% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: int8
   [NODATA] Using nodata value -128 for int8 data
   [PREDICTOR] Data type: int8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpr836x02a_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpk6e6sgz7.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpuned_EF50_WM_2024-10-03T23:31:16Z.tif
   [MEMORY] Final: 722.3 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpuned_EF50_WM_2024-10-03T23:31:16Z.tif

✅ Batch processing complete: 24 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 24
Successful: 24
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T01:21:47.729859


In [14]:
keys


['drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240914_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240919_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240921_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240926_N_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_20240926_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240914T233731_DVR_RTC20_G_gpufed_2C0F_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240914T233756_DVR_RTC20_G_gpufed_9ED7_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232218_DVR_RTC20_G_gpufed_9521_rgb.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232244_DVR_RTC20_G_gpufed_C767_WM.tif',
 'drcs_activations/202409_Hurricane_Helene/sentinel1/S1A_IW_20240916T232

In [32]:
print(len(keys))

46


In [12]:
def create_cog_filename_rgb(f, EVENT_NAME):
    """Create COG filename for RGB files, handling both simple and complex formats."""
    f2 = Path(f).stem
    fsplit = f2.split('_')
    
    # Check if it's a simple format (S1A_YYYYMMDD_rgb) or has *N* variant
    if len(fsplit) <= 4 and fsplit[0] == 'S1A':
        # Simple format
        if len(fsplit) == 3:  # S1A_20240914_rgb
            date_str = fsplit[1]
            # Format date from YYYYMMDD to YYYY-MM-DD
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
            cog_filename = f'{EVENT_NAME}_S1A_rgb_{formatted_date}_day.tif'
        elif len(fsplit) == 4 and fsplit[2] == 'N':  # S1A_20240926_N_rgb
            date_str = fsplit[1]
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
            cog_filename = f'{EVENT_NAME}_S1A_N_rgb_{formatted_date}_day.tif'
        else:
            # Fallback
            cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    
    # Complex format with timestamp (S1A_IW_YYYYMMDDTHHMMSS_...)
    elif len(fsplit) >= 8 and 'T' in fsplit[2]:
        # Extract components
        satellite = fsplit[0]
        mode = fsplit[1]
        datetime_str = fsplit[2]  # YYYYMMDDTHHMMSS
        
        # Use convert_sentinel_datetime for full timestamp
        formatted_datetime = convert_sentinel_datetime(datetime_str)
        
        # Get processing info
        processing_info = '_'.join(fsplit[3:8])  # DVR_RTC20_G_gpufed_9521
        
        cog_filename = f'{EVENT_NAME}_{satellite}_{mode}_{processing_info}_rgb_{formatted_datetime}.tif'
    
    else:
        # Fallback for any other format
        cog_filename = f'{EVENT_NAME}_{f2}_day.tif'
    
    return cog_filename

filter_str = 'rgb'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
print(len(filter_))

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_rgb(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")
    


Testing WM filename:
22
  202409_Hurricane_Helene_S1A_rgb_2024-09-14_day.tif
  202409_Hurricane_Helene_S1A_rgb_2024-09-19_day.tif
  202409_Hurricane_Helene_S1A_rgb_2024-09-21_day.tif
  202409_Hurricane_Helene_S1A_N_rgb_2024-09-26_day.tif
  202409_Hurricane_Helene_S1A_rgb_2024-09-26_day.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9521_rgb_2024-09-16T23:22:18Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_C767_rgb_2024-09-16T23:22:44Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_FCE1_rgb_2024-09-16T23:23:09Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_4389_rgb_2024-09-16T23:23:34Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_CCA1_rgb_2024-09-28T23:22:18Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_E205_rgb_2024-09-28T23:22:44Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_75C4_rgb_2024-09-28T23:23:09Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_2CC8_rgb_2024-09-28T23:23:34Z.tif
  202409_Hurricane_He

In [13]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_rgb, 
                                target_dir = "Sentinel-1/rgb", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202409_Hurricane_Helene_S1A_rgb_2024-09-14_day.tif
  202409_Hurricane_Helene_S1A_rgb_2024-09-19_day.tif
  202409_Hurricane_Helene_S1A_rgb_2024-09-21_day.tif
  202409_Hurricane_Helene_S1A_N_rgb_2024-09-26_day.tif
  202409_Hurricane_Helene_S1A_rgb_2024-09-26_day.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_9521_rgb_2024-09-16T23:22:18Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_C767_rgb_2024-09-16T23:22:44Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_FCE1_rgb_2024-09-16T23:23:09Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_4389_rgb_2024-09-16T23:23:34Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_CCA1_rgb_2024-09-28T23:22:18Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_E205_rgb_2024-09-28T23:22:44Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_75C4_rgb_2024-09-28T23:23:09Z.tif
  202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpufed_2CC8_rgb_2024-09-28T23:23:34Z.tif
  202409_Hurricane_Helene_

Band 1:  13%|█▎        | 66/494 [00:02<00:19, 21.58chunks/s]


   [MEMORY] High usage: 603.8 MB, forcing cleanup...


Band 1:  16%|█▌        | 77/494 [00:03<00:18, 22.08chunks/s]


   [MEMORY] High usage: 624.2 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=17, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp1xys2y1c_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp_b28b4s_.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpuned_EF50_rgb_2024-10-03T23:31:16Z.tif
   [MEMORY] Final: 1441.3 MB (Change: +49.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_S1A_IW_DVR_RTC20_G_gpuned_EF50_rgb_2024-10-03T23:31:16Z.tif

✅ Batch processing complete: 22 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 22
Successful: 22
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T02:55:28.

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")